## Notebook 04: Model Training, Cross-Validation & Model Selection

---

### 1. Architectural Role & Project Context
This notebook serves as the core training and validation pipeline for the disaster response classifier. In mission-critical emergency support, models must achieve high discriminatory power across all priority classes without overfitting to common scenarios or suffering from data leakage.

Key objectives:
1. Fit a leak-free scikit-learn preprocessing pipeline on training scenarios exclusively.
2. Train and tune **Random Forest** and **XGBoost** multi-class classifiers.
3. Benchmark models using **5-Fold Stratified Cross-Validation Macro F1**.
4. Serialize champion model, preprocessor, and metadata to `models/` for downstream inference.

---

### 2. Linked Source Scripts (`src/component04_ml/`)
- **`src/component04_ml/model_training.py`**:
  - `train_decision_models(train_df, val_df, config, models_dir, ...)` (or `train_models`): Orchestrates preprocessor fitting, model training, stratified cross-validation, champion selection, and artifact serialization.
- **`src/component04_ml/preprocessing.py`**:
  - `DecisionDataPreprocessor`: Implements leakage-safe imputation, standard scaling, and one-hot encoding into a serializable Scikit-Learn transformer.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.component04_ml.model_training import train_decision_models

train_path = root_dir / "data" / "processed" / "train_scenarios.csv"
models_dir = root_dir / "models"


### Section 1: Execute Training Pipeline with 5-Fold Stratified Cross-Validation
**Source Module:** `src/component04_ml/model_training.py` -> `train_decision_models()`

#### Pipeline Step-by-Step Architecture:
1. **Preprocessor Fitting**:
   - `DecisionDataPreprocessor` fits `SimpleImputer(strategy='median')` and `StandardScaler()` on 13 numerical features (including engineered indices).
   - Fits `OneHotEncoder(handle_unknown='ignore')` on 4 categorical features (`contamination_risk_category`, `infrastructure_importance`, `communication_status`, `catchment_zone`).
   - Serializes fitted pipeline to `models/preprocessor.joblib`.
2. **Random Forest Classifier**:
   - 150 estimators, max depth 12, min samples split 5, min samples leaf 2.
   - Robust ensemble bagging to control variance across noisy sensor scenarios.
3. **XGBoost Classifier**:
   - 150 gradient-boosted trees, max depth 6, learning rate 0.08, subsample 0.85, colsample_bytree 0.85.
   - Multi-class softprob objective (`num_class=4`, `mlogloss` metric).
4. **Stratified 5-Fold Cross-Validation**:
   - Computes Macro F1 on each out-of-fold split to ensure fair evaluation across minority critical classes (P1).


In [2]:
results = train_decision_models(
    train_data_path=train_path,
    output_models_dir=models_dir,
    random_seed=42,
    cv_folds=5
)


### Section 2: Compare Cross-Validation Results

#### Validation Metric: Macro-Averaged F1-Score
In disaster response, **Macro F1** is the primary evaluation metric because it treats each priority level (P1, P2, P3, P4) with equal weight:
$$\text{Macro F1} = \frac{\text{F1}_{P1} + \text{F1}_{P2} + \text{F1}_{P3} + \text{F1}_{P4}}{4}$$
This prevents high accuracy on frequent low-priority events from masking dangerous false negatives on rare P1 emergencies.


In [3]:
cv_f1 = results["metadata"]["cv_macro_f1"]

comparison_df = pd.DataFrame({
    "Candidate Model": ["Random Forest Classifier", "XGBoost Classifier"],
    "5-Fold CV Macro F1": [
        cv_f1["random_forest"],
        cv_f1["xgboost"]
    ]
})

print("Cross-Validation Macro F1 Performance:")
comparison_df


Cross-Validation Macro F1 Performance:


,Candidate Model,5-Fold CV Macro F1
0,Random Forest Classifier,0.9208
1,XGBoost Classifier,0.9385


### Section 3: Model Selection & Artifact Inspection

#### Champion Selection Rationale:
- The classifier achieving the highest 5-Fold CV Macro F1 is designated as the primary production model and saved as `models/selected_model.joblib`.
- Complete model hyperparameters and performance metrics are logged in `models/model_metadata.json`.


In [4]:
print(f"Selected Champion Model: {results['selected_name']}")
print("Serialized artifacts in models/:")
for p in models_dir.glob("*.joblib"):
    print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")


Selected Champion Model: xgboost_classifier
Serialized artifacts in models/:
  preprocessor.joblib (7.5 KB)
  random_forest_model.joblib (5509.2 KB)
  selected_model.joblib (1187.7 KB)
  xgboost_model.joblib (1187.7 KB)
